# Collecte des événements culturels — Open Agenda (Rennes)

**Objectif :** récupérer tous les événements culturels à Rennes via l'API publique OpenDataSoft,  
filtrer sur moins d'un an, et sauvegarder le résultat en CSV dans `data/raw/`.

**Source :** https://public.opendatasoft.com/explore/dataset/evenements-publics-openagenda  
**Pas de clé API requise** — données publiques.

In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

## 1. Exploration de l'API

Avant de tout récupérer, on fait un premier appel avec `limit=1` pour examiner la structure d'un enregistrement.  
C'est essentiel pour savoir quels champs existent et lesquels sont utiles pour notre RAG.

In [2]:
BASE_URL = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records"

response = requests.get(BASE_URL, params={
    "where": 'location_city="Rennes"',
    "limit": 1
})

response.raise_for_status()
data = response.json()

print(f"Total disponible : {data['total_count']}")

import json
print(json.dumps(data['results'][0], indent=2, ensure_ascii=False))

Total disponible : 17360
{
  "uid": "18898679",
  "slug": "voyage-dans-le-temps-le-parc-du-thabor",
  "canonicalurl": "https://openagenda.com/rdvj-2024-bretagne/events/voyage-dans-le-temps-le-parc-du-thabor",
  "title_fr": "Voyage dans le temps : le parc du Thabor",
  "description_fr": "A l'occasion de l'opération \"Rendez-vous aux jardins\", retour au XIXe siècle en compagnie de votre guide costumé pour découvrir les usages de ce remarquable parc aménagé aupar le sfrères Bülher : ses…",
  "longdescription_fr": "<p>A l'occasion de l'opération \"Rendez-vous aux jardins\", retour au XIXe siècle en compagnie de votre guide costumé pour découvrir les usages de ce remarquable parc aménagé aupar le sfrères Bülher : ses arhitectures, les jardins à la française ou pittoresques, l' \"Enfer\", le jardin botanique et la célèbre roseraie...</p>",
  "conditions_fr": "Tarif plein : 9, 50 €, nombre de places limitées",
  "keywords_fr": null,
  "image": "https://cibul.s3.amazonaws.com/cd1f6f74bcc9423b

## 2. Analyse de la structure

Tous les champs collectés alimenteront le corpus RAG (texte embedé et recherché) :

- `title_fr` — titre
- `longdescription_fr` — description complète
- `conditions_fr` — tarif
- `firstdate_begin`, `lastdate_end` — dates
- `location_name`, `location_address` — lieu
- `canonicalurl` — lien vers l'événement

Le champ `uid` est conservé uniquement comme identifiant technique.

## 3. Filtre temporel

On ajoute un filtre `lastdate_end >= aujourd'hui - 1 an` pour ne garder que les événements récents.  
On vérifie d'abord combien d'événements passent ce filtre avant de tout télécharger.

In [3]:
date_limit = (datetime.now(timezone.utc) - timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%SZ")

response = requests.get(BASE_URL, params={
    "where": f'location_city="Rennes" AND lastdate_end >= "{date_limit}"',
    "limit": 1
})

response.raise_for_status()
print(f"Événements dans la fenêtre d'un an : {response.json()['total_count']}")

Événements dans la fenêtre d'un an : 5872


## 4. Collecte complète avec pagination

L'API limite à 100 résultats par requête. On boucle en incrémentant `offset` jusqu'à avoir tout récupéré.

In [4]:
FIELDS = "uid,title_fr,longdescription_fr,conditions_fr,firstdate_begin,lastdate_end,location_name,location_address,canonicalurl"

all_records = []
offset = 0
total = 5745

while offset < total:
    response = requests.get(BASE_URL, params={
        "where": f'location_city="Rennes" AND lastdate_end >= "{date_limit}"',
        "select": FIELDS,
        "limit": 100,
        "offset": offset
    })
    response.raise_for_status()
    records = response.json()["results"]
    all_records.extend(records)
    offset += 100
    print(f"{len(all_records)}/{total}", end="\r")

print(f"\nCollecte terminée : {len(all_records)} événements")

5800/5745
Collecte terminée : 5800 événements


## 5. Sauvegarde en CSV

In [5]:
df = pd.DataFrame(all_records)

print(df.shape)
print(df.dtypes)
df.head(3)

(5800, 9)
uid                   object
title_fr              object
longdescription_fr    object
conditions_fr         object
firstdate_begin       object
lastdate_end          object
location_name         object
location_address      object
canonicalurl          object
dtype: object


,uid,title_fr,longdescription_fr,conditions_fr,firstdate_begin,lastdate_end,location_name,location_address,canonicalurl
0,51417014,comment se former dans l'agroalimentaire ?,<p>Venez découvrir les solutions de formations...,None,2025-11-06T09:00:00+00:00,2025-11-06T10:00:00+00:00,Rennes - BRETAGNE,35000 Rennes,https://openagenda.com/semaine-industrie-2025/...
1,16397447,Le Quart d'Heure Info,<p>L'actualité de l'emploi abordée en 15 min</...,None,2025-09-01T12:00:00+00:00,2025-09-01T12:15:00+00:00,Agence RENNES NORD,35700 Rennes,https://openagenda.com/francetravail/events/le...
2,83202023,Les Rennaises sous la Seconde Guerre mondiale,<p>C’est par l’évocation de personnalités fémi...,None,2025-09-20T08:00:00+00:00,2025-09-20T14:30:00+00:00,Destination Rennes - Office de Tourisme,1 rue de Saint-Malo 35000 Rennes,https://openagenda.com/culture/events/les-renn...


In [6]:
df.to_csv("../data/raw/events.csv", index=False, encoding="utf-8")
print("Sauvegardé dans data/raw/events.csv")

Sauvegardé dans data/raw/events.csv
